[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/memorization_generalization_in_diffusion_models_tests/blob/main/tp/TP_extraction_memorisation.ipynb)

# Retrouver les images mémorisées par un modèle de diffusion

Les débruiteurs CelebA des auteurs ont été entraînés sur des sous-ensembles dont on ignore la composition. Si un modèle mémorise, ses échantillons sont des copies de ses images d'entraînement : en cherchant leurs plus proches voisins dans CelebA, on **retrouve** ces images (principe des attaques par extraction de données, Carlini et al., 2023), ce qui permet ensuite de tracer les vraies courbes de débruitage sur l'entraînement et sur le test (figure 1 de l'article).

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter la cellule d'installation.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "memorization_generalization_in_diffusion_models_tests"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/code/',
            '/tp/tp_outils.py',
            '/tp/prepare_celeba80x80.py',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_1/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_1_swapped/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_10/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_10_swapped/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_100/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_100_swapped/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_1000/',
            '/denoisers/UNet/img_align_celeba_80x80/0to255_RF_90x90_set_size_1000_swapped/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    racine = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(racine, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), racine], check=True)
    subprocess.run(["git", "-C", racine, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", racine, "checkout", BRANCH], check=True)
    try:
        import pywt   # PyWavelets (générateur d'images C^alpha des auteurs) : absent de certaines images Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "code", "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
os.makedirs(os.path.join(racine, "tp"), exist_ok=True)   # absent du clone partiel
os.chdir(os.path.join(racine, "tp"))
for d in ("code", "tp"):
    if os.path.join(racine, d) not in sys.path:
        sys.path.insert(0, os.path.join(racine, d))
print("Répertoire courant :", os.getcwd())

## 0. Réglages

Prévu pour un GPU (Colab) : une vingtaine de minutes (non vérifié), plus la reconstruction de CelebA (environ
1,4 Go, 5 à 10 minutes) au premier lancement. Sur CPU, le nombre d'échantillons est réduit automatiquement
(environ 11 minutes, CelebA déjà reconstruit).

In [ ]:
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_extraction_memorisation")   # figures nommées dans resultats/figures/

GRAINE = 0
N_LISTE = [1, 10, 100, 1000]
if T.DEVICE.type == "cuda":
    N_ECHANTILLONS = {1: 16, 10: 64, 100: 256, 1000: 256}
else:
    N_ECHANTILLONS = {1: 8, 10: 32, 100: 64, 1000: 64}
SIGMA_FIN = 0.05        # valeur de l'article (0,1 dans le notebook des auteurs)
SEUIL = 0.95            # similarité cosinus au-delà de laquelle deux images sont tenues pour identiques
print("calcul sur", T.DEVICE)

## 1. Données et modèles

CelebA reconstruit (198 650 images 80×80) et les débruiteurs du groupe A (entraînés sur un sous-ensemble
$S_1$) et du groupe B (sous-ensemble disjoint $S_2$).

In [ ]:
donnees = T.charger_celeba()
moyenne = donnees.mean(dim=0)
print("CelebA reconstruit :", tuple(donnees.shape))
modeles = {(N, g): T.charger_unet_auteurs("img_align_celeba_80x80", N, echange=(g == "B"))
           for N in N_LISTE for g in "AB"}
print(len(modeles), "débruiteurs chargés")

## 2. Échantillons

Échantillonneur de l'article (`T.echantillonner`). L'initialisation est la moyenne de **tout** CelebA plus
un bruit blanc d'écart-type 1 : les auteurs utilisent la moyenne du jeu d'entraînement du modèle, que l'on
ne connaît pas. Les échantillons qui divergent (valeurs hors de $[-0{,}5;\ 1{,}5]$) sont écartés et comptés.

In [ ]:
echantillons = {}
for N in N_LISTE:
    init = T.init_echantillons(moyenne, N_ECHANTILLONS[N], graine=GRAINE)
    ech, infos = T.echantillonner(modeles[(N, "A")], init, sigma_fin=SIGMA_FIN, graine=GRAINE)
    ech = ech.cpu()
    valides = (ech.amin(dim=(1, 2, 3)) > -0.5) & (ech.amax(dim=(1, 2, 3)) < 1.5)
    echantillons[N] = ech[valides]
    print("N = %4d : %3d échantillons, %3d divergents écartés, %d itérations au plus"
          % (N, len(ech), (~valides).sum(), infos["iterations"]))
for N in N_LISTE:
    T.afficher_images(echantillons[N][:10], titre="échantillons du modèle A, N = %d" % N, nom="echantillons_N%d" % N)

## 3. Repérer la mémorisation sans les données : la diversité des échantillons

On regroupe les échantillons presque identiques (similarité cosinus supérieure à `SEUIL`). Un modèle qui
mémorise $N$ images ne peut produire plus de $N$ groupes.

In [ ]:
for N in N_LISTE:
    groupes = T.groupes_distincts(echantillons[N], SEUIL)
    print("N = %4d : %3d échantillons -> %3d groupes distincts" % (N, len(groupes), groupes.max() + 1))

## 4. Extraction : plus proches voisins dans CelebA

Pour chaque échantillon, l'image de CelebA la plus semblable (similarité cosinus, images centrées).

In [ ]:
extraits = {}
fig, axes = plt.subplots(1, len(N_LISTE), figsize=(4 * len(N_LISTE), 3.2), sharey=True)
for ax, N in zip(axes, N_LISTE):
    sim, idx = T.plus_proches_voisins(echantillons[N], donnees)
    extraits[N] = (sim, idx)
    ax.hist(sim.numpy(), bins=np.linspace(0, 1, 51))
    ax.axvline(SEUIL, color="k", ls=":")
    ax.set_title("N = %d" % N); ax.set_xlabel("similarité au plus proche voisin")
axes[0].set_ylabel("nombre d'échantillons")
T.montrer("similarites")
for N in N_LISTE:
    sim, idx = extraits[N]
    retrouvees = sorted(set(idx[sim > SEUIL].tolist()))
    print("N = %4d : %3d images retrouvées (similarité > %.2f) ; positions dans CelebA reconstruit : %s"
          % (N, len(retrouvees), SEUIL, retrouvees[:12]))
N = 10
sim, idx = extraits[N]
T.afficher_images(torch.cat([echantillons[N][:10], donnees[idx[:10]]]), ncol=10,
                  titre="N = 10 : échantillons (en haut) et plus proches voisins dans CelebA (en bas)", nom="voisins_N10")

## 5. Les vraies courbes de débruitage entraînement / test

Les images retrouvées sont des images d'entraînement du modèle A, et des images de **test** pour le modèle B
(entraîné sur un sous-ensemble disjoint). On compare aussi les deux modèles sur 300 images qui ne sont pas
parmi les images retrouvées.

In [ ]:
sig_range = np.logspace(0.3, 2.4, 10) / 255
g = torch.Generator().manual_seed(GRAINE)
autres = donnees[torch.randperm(len(donnees), generator=g)[:300]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
courbes_train, courbes_test = {}, {}
for N in N_LISTE:
    sim, idx = extraits[N]
    retrouvees = donnees[sorted(set(idx[sim > SEUIL].tolist()))]
    if len(retrouvees) == 0:
        continue
    fA = lambda y, s, m=modeles[(N, "A")]: y - m(y)
    fB = lambda y, s, m=modeles[(N, "B")]: y - m(y)
    courbes_train["N = %d (%d images)" % (N, len(retrouvees))] = T.courbe_psnr(fA, retrouvees, sig_range)
    courbes_test["N = %d, modèle B" % N] = T.courbe_psnr(fB, retrouvees, sig_range)
T.trace_psnr(courbes_train, sig_range, "Modèle A sur ses images retrouvées (entraînement)", ax=axes[0])
T.trace_psnr(courbes_test, sig_range, "Modèle B sur les mêmes images (test)", ax=axes[1])
T.montrer("psnr")
for nom, v in courbes_train.items():
    print("entraînement %-22s %s" % (nom, np.round(v, 1)))
for nom, v in courbes_test.items():
    print("test         %-22s %s" % (nom, np.round(v, 1)))